In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
from PIL import ImageFile
import pandas as pd
import numpy as np
import shutil
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [3]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

modelRestnet = models.Sequential()
modelRestnet.add(base_model)
modelRestnet.add(layers.GlobalMaxPooling2D())
modelRestnet.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 2048)           │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# train + vali

In [5]:
trainValiPath='/content/gdrive/MyDrive/ml/train+vali'
trainValiImage = datagen.flow_from_directory(trainValiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
trainValifeatures = modelRestnet.predict(trainValiImage, verbose=1)

Found 2954 images belonging to 18 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


93/93 ━━━━━━━━━━━━━━━━━━━━ 1476s 16s/step


In [6]:
trainValifilenames = trainValiImage.filenames
trainValifeatures = trainValifeatures / np.linalg.norm(trainValifeatures, axis=1, keepdims=True)
df_trainVali = pd.DataFrame(trainValifeatures)
df_trainVali.insert(0, "filename", trainValifilenames)
df_trainVali.to_csv("trainValiFeature.csv", index=False)

In [7]:
df_trainVali.to_csv("trainValiFeature.csv", index=False)
shutil.move('trainValiFeature.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')

'/content/gdrive/MyDrive/ml/restnet50Finetune/trainValiFeature.csv'

# seperate train and vali

In [8]:
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [9]:
trainImage = datagen.flow_from_directory(trainPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )
valiImage= datagen.flow_from_directory(valiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [10]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [11]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [12]:
train_features = modelRestnet.predict(trainImage, verbose=1)
val_features = modelRestnet.predict(valiImage, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1284s 14s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 323s 14s/step


In [13]:
train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

train_features = train_features / np.linalg.norm(train_features, axis=1, keepdims=True)
val_features = val_features / np.linalg.norm(val_features, axis=1, keepdims=True)

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [14]:
import joblib

filename = 'restnet50Finetune.sav'
joblib.dump(modelRestnet, filename)

['restnet50Finetune.sav']

In [15]:
shutil.move('restnet50Finetune.sav', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('trainFeature.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('trainLabel.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('valiLabel.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('valiFeature.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')

'/content/gdrive/MyDrive/ml/restnet50Finetune/valiFeature.csv'

# End of Restnet50

In [16]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [17]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50Finetune/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/restnet50Finetune/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [18]:
from sklearn.preprocessing import StandardScaler
stdscaler = StandardScaler()
stdscaler.fit(trainFeature)

trainFeature = stdscaler.transform(trainFeature)
valiFeature = stdscaler.transform(valiFeature)

In [19]:
trainLabel= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50Finetune/trainLabel.csv')
valiLabel = pd.read_csv('/content/gdrive/MyDrive/ml/restnet50Finetune/valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [20]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

trainLabel = trainLabel.astype(int)
valiLabel = valiLabel.astype(int)

In [21]:
from tensorflow.keras.optimizers import Adam

In [22]:
model = models.Sequential()
model.add(layers.Dense(512, input_shape=(2048,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.0001)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 18)             │         9,234 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,325,074 (5.05 MB)

 Trainable params: 1,323,026 (5.05 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [23]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [24]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.3356 - loss: 2.2997 - val_accuracy: 0.8660 - val_loss: 0.6799
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8621 - loss: 0.6614 - val_accuracy: 0.9409 - val_loss: 0.3281
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9390 - loss: 0.3467 - val_accuracy: 0.9685 - val_loss: 0.1936
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9642 - loss: 0.2361 - val_accuracy: 0.9895 - val_loss: 0.1258
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9795 - loss: 0.1718 - val_accuracy: 0.9947 - val_loss: 0.0856
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9893 - loss: 0.1222 - val_accuracy: 0.9947 - val_loss: 0.0614
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9917 - loss: 0.0958 - val_accuracy: 0.9961 - val_loss: 0.0474
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0729 - val_accuracy: 0.9961 - 

In [25]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())
new_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,315,840 (5.02 MB)

 Trainable params: 1,313,792 (5.01 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [26]:
joblib.dump(new_model, 'cnnRestnet50Finetune.sav')

['cnnRestnet50Finetune.sav']

# Train done

In [27]:
import joblib
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [28]:
# stdscaler = joblib.load('/content/gdrive/MyDrive/ml/restnet50Finetune/stdScaler.pkl')
# new_model = joblib.load('/content/gdrive/MyDrive/ml/restnet50Finetune/cnnRestnet50Finetune.sav')

In [29]:
trainValiFeature= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50Finetune/trainValiFeature.csv')
trainValiFilename=trainValiFeature['filename']
trainValiFeature=trainValiFeature.drop(['filename'],axis=1)

In [30]:
trainValiFeatures = stdscaler.transform(trainValiFeature.values)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [31]:
trainValiFeatures[0]

array([ 0.41174167, -0.13182549, -1.01853902, ..., -0.84005584,
       -0.92497952, -0.3433297 ])

In [32]:
trainVali = new_model.predict(trainValiFeatures, verbose=1)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [33]:
trainVali.shape

(2954, 512)

In [34]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(trainVali)

MinMaxScaler(feature_range=(-1, 1))

In [35]:
trainVali[0]

array([5.52346110e-01, 1.31774783e-01, 6.06465697e-01, 4.56447721e-01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 3.71486574e-01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 7.69383490e-01, 0.00000000e+00, 7.80701637e-01,
       3.49414498e-01, 3.19762468e-01, 0.00000000e+00, 3.93457562e-01,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 7.77028680e-01,
       1.50837231e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 1.14284945e+00, 1.75404382e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 9.73086715e-01, 8.91617298e-01,
       2.40479064e+00, 1.30040562e+00, 0.00000000e+00, 1.28911841e+00,
       0.00000000e+00, 2.87186921e-01, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 8.84774625e-01, 0.00000000e+00, 0.00000000e+00,
       9.47567821e-03, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
      

In [36]:
trainVali = scaler.transform(trainVali)

In [37]:
trainVali[0]

array([-0.7297059 , -0.9313645 , -0.6663742 , -0.7370231 , -1.        ,
       -1.        , -1.        , -0.7662053 , -1.        , -1.        ,
       -1.        , -1.        , -1.        , -0.43986475, -1.        ,
       -0.42752755, -0.8103435 , -0.80024344, -1.        , -0.7761826 ,
       -1.        , -1.        , -1.        , -0.5195849 , -0.13968682,
       -1.        , -1.        , -1.        , -1.        , -0.34670335,
        0.20586693, -1.        , -1.        , -1.        , -0.40622222,
       -0.36858338,  0.30911863, -0.18833727, -1.        , -0.1314506 ,
       -1.        , -0.8105026 , -1.        , -1.        , -1.        ,
       -1.        , -1.        , -1.        , -1.        , -0.4408316 ,
       -1.        , -1.        , -0.9940169 , -1.        , -1.        ,
       -1.        , -0.42248237, -1.        , -1.        , -1.        ,
       -1.        , -0.9238438 , -1.        , -1.        , -1.        ,
       -0.7761702 , -1.        , -1.        , -1.        , -1.  

In [38]:
df_trainVali = pd.DataFrame(trainVali)
df_trainVali.insert(0, "filename", trainValiFilename)

In [39]:
joblib.dump(scaler, 'minMaxScaler.pkl')

['minMaxScaler.pkl']

In [40]:
import joblib
df_trainVali.to_csv("trainValiVectors.csv", index=False)
joblib.dump(scaler, 'minMaxScaler.pkl')
joblib.dump(new_model, 'cnnRestnet50Finetune.sav')
joblib.dump(stdscaler, 'stdScaler.pkl')

['stdScaler.pkl']

In [41]:
shutil.move('cnnRestnet50Finetune.sav', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('minMaxScaler.pkl', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('stdScaler.pkl', '/content/gdrive/MyDrive/ml/restnet50Finetune/')
shutil.move('trainValiVectors.csv', '/content/gdrive/MyDrive/ml/restnet50Finetune/')

'/content/gdrive/MyDrive/ml/restnet50Finetune/trainValiVectors.csv'